

# Acknowledgements and Declarations
>- The number before each code block corresponds to the relevant section in the report.
>- I would like to clarify the references in this project. 
I maintained some same variable naming as Ryan et al. (2025)
to get clear communication and direct comparison.


In [1]:
from pathlib import Path
from datetime import datetime
import zipfile
import matplotlib.pyplot as plt
import pandas as pd
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import (precision_score, recall_score, roc_auc_score, accuracy_score, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
# Imblearn Pipeline and Resampling
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
#Deepleaning parts
import torch
from torch import nn
from skorch import NeuralNetBinaryClassifier
from sklearn.metrics import roc_curve, classification_report
import shap
import pickle
from skorch.callbacks import EarlyStopping


## Paths

In [2]:
DATA_ZIP = "./data/covid-19-tracker-master/covid-19-tracker-master/data/australia.zip"
DATA_CSV = "./raw_data/australia.csv"
POLICY_CSV = "./data/covid-policy-tracker-master/covid-policy-tracker-master/data/Australia/OxCGRT_AUS_latest.csv"
DOSES_CSV = "./data/At least 3 doses.csv"
CASES_CSV = "./data/cases_daily_state.csv"
OUTPUT_DIR = Path("./data/generated_outputs")
MODEL_DIR = Path("./models")
RESULTS_DIR = Path("./results")
FIG_DIR = Path("./figures")
for d in [MODEL_DIR, RESULTS_DIR, FIG_DIR]:
    d.mkdir(exist_ok=True, parents=True)

## Load Australia survey data

In [3]:
def read_australia_csv_from_zip(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as z:
        csv_names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise FileNotFoundError("No CSV found inside australia.zip")
        with z.open(csv_names[0]) as f:
            return pd.read_csv(
                f,
                na_values=[" ", "__NA__"],
                keep_default_na=True,
                low_memory=False,
            )

def load_australia_data() -> pd.DataFrame:
    if Path(DATA_ZIP).exists():
        return read_australia_csv_from_zip(Path(DATA_ZIP))
    if Path(DATA_CSV).exists():
        return pd.read_csv(
            DATA_CSV,
            na_values=[" ", "__NA__"],
            keep_default_na=True,
            low_memory=False,
        )
    raise FileNotFoundError("Could not find australia.zip or australia.csv")

raw_df = load_australia_data()
raw_df.tail()

,RecordNo,endtime,qweek,i1_health,i2_health,i7a_health,i3_health,i4_health,i5_health_1,i5_health_2,...,vac_man_99,q_other,household_children_resp,had_covid,vac_boost_beyond,future_1,future_2,had_covid_2,long_covid,future_3
53828,53828,28/03/2022 09:45,week 54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,NaN,0,NaN,3,3 – About the same,3 – About the same,No,NaN,3 – About the same
53829,53829,28/03/2022 09:47,week 54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Yes,NaN,1,NaN,2,3 – About the same,3 – About the same,No,NaN,3 – About the same
53830,53830,28/03/2022 09:53,week 54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Yes,NaN,0,NaN,4,3 – About the same,2,No,NaN,1 – A lot less impact
53831,53831,28/03/2022 10:07,week 54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,NaN,0,NaN,NaN,3 – About the same,3 – About the same,No,NaN,3 – About the same
53832,53832,28/03/2022 10:19,week 54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,NaN,0,NaN,2,3 – About the same,4,No,NaN,2


## Missing_table

In [4]:
missing_value_df = pd.DataFrame(
    [(col, raw_df[col].isna().sum()) for col in raw_df.columns],
    columns=["Variable Name", "Missing Value Count"],
).sort_values(by=["Missing Value Count", "Variable Name"]).reset_index(drop=True)

missing_value_df.to_csv(OUTPUT_DIR / "missing_value_counts.csv", index=False)
missing_value_df

,Variable Name,Missing Value Count
0,RecordNo,0
1,age,0
2,employment_status,0
3,endtime,0
4,gender,0
...,...,...
508,V3_me_other,53831
509,V3_baby,53832
510,m6_other,53832
511,V3_baby_other,53833


## 5.2 Cleaning and Processing
To meet the requirements of the model for input data, this study preprocessed the data based on Ryan et al.'s (2025) method. The specific feature conversion and cleaning steps are divided into eight core steps.
 The number in front of each code cell below indicates the corresponding section number in the report's text description.


### 5.2.1 Government Mandates
This section calculated the 14-day rolling average of each state’s index (daily updated 14-day average of each state’s index). When the rolling average of a certain state reached 3 for the first time, it was considered that the jurisdiction had actually entered a stable period of mandate intervention. The integration of mandate status can be found at the end of this chapter.

In [5]:
policy_df = pd.read_csv(POLICY_CSV)
policy_df = policy_df.loc[:, ["RegionName", "RegionCode", "Date", "H6M_Facial Coverings"]].copy()
policy_df.index = pd.to_datetime(policy_df["Date"], format="%Y%m%d")

#Calculate the 14 day rolling average of each state’s index
rolling_days = 14
df_rolling = policy_df.loc[:, ["RegionName", "H6M_Facial Coverings"]].groupby(
    "RegionName"
).rolling(window=rolling_days).mean()
# When the rolling average of a certain state reached 3 for the first time, it was considered that the jurisdiction had actually entered a stable period of mandatory intervention
mandate_limit = 3
mandate_start_df = df_rolling[df_rolling["H6M_Facial Coverings"] >= mandate_limit].groupby("RegionName").head(1)

mandate_start_df.to_csv(OUTPUT_DIR / "mandate_start_dates.csv")
mandate_start_df

,,H6M_Facial Coverings
RegionName,Date,
Australian Capital Territory,2021-08-18,3.000000
New South Wales,2021-07-09,3.000000
Northern Territory,2021-11-21,3.000000
Queensland,2021-01-17,3.142857
South Australia,2021-07-26,3.000000
Tasmania,2021-10-21,3.000000
Victoria,2020-07-21,3.000000
Western Australia,2021-02-08,3.000000


During initial Excel observation, two different dash characters were found.
Although the code environment can recognize both, I standardize them to a single format here to avoid potential bugs in downstream processing.

In [6]:
clean_df = raw_df.copy()

def norm_str(ser):
    return ser.astype(str).str.strip()

str_cols = ["r1_1", "r1_2"] + [f"i12_health_{i}" for i in range(1, 26)]
for c in str_cols:
    if c in clean_df.columns:
        clean_df[c] = norm_str(clean_df[c])

### 5.2.2 Removal of High Missing Features
Removal of High Missing Features. Also, I added the week number in this chapter

In [7]:
def convert_datetime(dt):
    if isinstance(dt, (pd.Timestamp, datetime)):
        return dt
    s = str(dt).strip().split()[0]
    return datetime.strptime(s, "%d/%m/%Y")

clean_df["endtime"] = clean_df["endtime"].apply(convert_datetime)

thresh_value = 10781
columns_to_drop = missing_value_df.loc[
    missing_value_df["Missing Value Count"] > thresh_value,
    "Variable Name",
].tolist()
clean_df.drop(columns=columns_to_drop, inplace=True)
start_date = clean_df["endtime"].min()
clean_df["week_number"] = ((clean_df["endtime"] - start_date).dt.days // 14) + 1

### 5.2.3 Repairing Data Gaps Caused by Survey Changes
Regarding the start date for addressing the data gap, following image analysis and personal correspondence with the original author (Ryan), it was confirmed that the February 19th date reported in the original paper was a typographical error. Therefore, February 10th is used as the start date in this analysis. For the imputation experiment, please refer to 6.2.1 in this notebook.

In [8]:
sdate, edate = "2021-02-10", "2021-10-18" 
consent_gap_mask = (clean_df["endtime"] <= edate) & (clean_df["endtime"] >= sdate)


phq_cols = [f"PHQ4_{i}" for i in range(1, 5)]
d1_cols = [f"d1_health_{i}" for i in list(range(1, 14)) + [98, 99]]
cols_to_fill = phq_cols + d1_cols

clean_df.loc[consent_gap_mask, cols_to_fill] = clean_df.loc[consent_gap_mask, cols_to_fill].fillna("N/A")
print(consent_gap_mask.sum())

18014


### 5.2.4 Processing of Predictive Variables
The core goals of the prediction are "self-protection behavior" and "mask wearing". The original text records the frequency of respondents performing a certain behavior in the past 7 days.

In [9]:
#Convert str to numbers
freq_to_score = {
    "Always": 5, "Frequently": 4, "Sometimes": 3, 
    "Rarely": 2, "Not at all": 1
}

i12_cols = [c for c in clean_df.columns if c.startswith("i12_health_")]
clean_df[i12_cols] = clean_df[i12_cols].replace(freq_to_score)

#Mask wearing behavior
mask_cols = ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]
clean_df["face_mask_scale"] = clean_df[mask_cols].median(axis=1)
clean_df["face_mask_bi"] = clean_df["face_mask_scale"].apply(
    lambda x: "Yes" if x >= 4 else ("No" if x < 4 else None)
)
#Comprehensive protective behavior
protective_behaviour_cols = [col for col in clean_df.columns if col.startswith("i12_")]
clean_df["protective_behaviour_scale"] = clean_df[protective_behaviour_cols].median(axis=1)
clean_df["protective_behaviour_bi"] = clean_df["protective_behaviour_scale"].apply(
    lambda x: "Yes" if x >= 4 else ("No" if x < 4 else None)
)
#Protective features other than masks
protective_behaviour_nomask_cols = [
    col
    for col in protective_behaviour_cols
    if col not in ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]
]
clean_df["protective_behaviour_nomask_scale"] = clean_df[
    protective_behaviour_nomask_cols
].median(axis=1)
clean_df["protective_behaviour_nomask_bi"] = clean_df["protective_behaviour_nomask_scale"].apply(
    lambda x: "Yes" if x >= 4 else ("No" if x < 4 else None)
)

C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\3236983135.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clean_df[i12_cols] = clean_df[i12_cols].replace(freq_to_score)


### 5.2.5 Dimensionality Reduction for Sparse Comorbidity Features
In the raw data, the comorbidities were scattered among 13 binary options (from d1_ health _1 to d1_ health _99), which makes the data extremely sparse. This study removed all specific disease types and made them into a single categorical variable called d1_comorbidities

In [10]:
def comorbidity(row):
    if row["d1_health_99"] == "Yes":
        return "No"
    elif row["d1_health_99"] == "N/A":
        return "N/A"
    elif row["d1_health_98"] == "Yes":
        return "Prefer_not_to_say"
    else:
        return "Yes"

clean_df["comorbidity_status"] = clean_df.apply(comorbidity, axis=1)
d1_cols = [c for c in clean_df.columns if c.startswith("d1_")]
clean_df = clean_df.drop(columns=d1_cols)

### 5.2.6 Processing of Demographic Variables
In order to reduce the effect of extreme values on trees or linear models, this study assigned a value of 8 to all samples with answers of "8 people or more".

In [11]:
household_mapping = {str(i): i for i in range(1, 8)}
household_mapping.update({
    "8 or more": 8, 
    "Prefer not to say": None, 
    "Don't know": None
})

clean_df["household_size"] = clean_df["household_size"].astype(str).str.strip().map(household_mapping)

### 5.2.7 Preprocessing of Perception, Trust, and Mental Health Variables 
Unordered categorical features was converted into dummy at the end of data processing.

In [12]:
# Perception of illness threat
r1_cols = ["r1_1", "r1_2"]
clean_df[r1_cols] = clean_df[r1_cols].replace({
    "7 - Agree": 7, "7 – Agree": 7, 
    "1 - Disagree": 1, "1 – Disagree": 1,
    "6": 6, "5": 5, "4": 4, "3": 3, "2": 2
}).apply(pd.to_numeric, errors="coerce")

clean_df.dropna(subset=r1_cols,inplace=True)

cleaned_data_df = clean_df.drop(["qweek", "weight"] + protective_behaviour_cols, axis=1).copy()
cleaned_data_df.to_csv(OUTPUT_DIR / "cleaned_data.csv", index=False, date_format="%Y-%m-%d")

cleaned_data_df.head()

,RecordNo,endtime,i2_health,i9_health,i11_health,age,gender,state,household_size,employment_status,...,r1_1,r1_2,week_number,face_mask_scale,face_mask_bi,protective_behaviour_scale,protective_behaviour_bi,protective_behaviour_nomask_scale,protective_behaviour_nomask_bi,comorbidity_status
9023,9023,2020-06-24,0.0,Not sure,Not sure,31,Male,Western Australia,1.0,Full time employment,...,5.0,1.0,7,3.0,No,3.0,No,3.0,No,Yes
9024,9024,2020-06-24,2.0,No,Very willing,36,Male,Victoria,4.0,Full time employment,...,6.0,4.0,7,4.0,Yes,3.0,No,3.0,No,No
9025,9025,2020-06-24,6.0,Yes,Very willing,73,Male,Northern Territory,2.0,Retired,...,5.0,6.0,7,1.0,No,4.0,Yes,5.0,Yes,Yes
9026,9026,2020-06-24,20.0,Yes,Somewhat willing,58,Male,Queensland,2.0,Not working,...,1.0,4.0,7,1.0,No,1.0,No,1.0,No,Yes
9027,9027,2020-06-24,0.0,Yes,Very willing,65,Male,Victoria,1.0,Full time employment,...,3.0,1.0,7,1.0,No,1.0,No,1.0,No,No


### 5.2.8 Merging Vaccination and Daily Cases Data

In [13]:
model_df = cleaned_data_df.copy()
df_doses = pd.read_csv(DOSES_CSV)
df_cases = pd.read_csv(CASES_CSV)

# Convert wide format to long format
doses_long = df_doses.melt(id_vars=["Date"], var_name="state_abbr", value_name="at_least_3_doses")
cases_long = df_cases.melt(id_vars=["Date"], var_name="state_abbr", value_name="daily_cases")

doses_long["at_least_3_doses"] = doses_long["at_least_3_doses"].astype(str).str.replace(",", "", regex=False)
doses_long["at_least_3_doses"] = pd.to_numeric(doses_long["at_least_3_doses"], errors="coerce")
doses_long["Date"] = pd.to_datetime(doses_long["Date"], format="%d/%m/%y")
cases_long["Date"] = pd.to_datetime(cases_long["Date"], format="%d/%m/%y")
model_df["endtime"] = pd.to_datetime(model_df["endtime"])

state_mapping = {
    "NSW": "New South Wales",
    "VIC": "Victoria",
    "QLD": "Queensland",
    "SA": "South Australia",
    "WA": "Western Australia",
    "TAS": "Tasmania",
    "NT": "Northern Territory",
    "ACT": "Australian Capital Territory"
}
doses_long["state"] = doses_long["state_abbr"].map(state_mapping)
cases_long["state"] = cases_long["state_abbr"].map(state_mapping)

# Merge vaccination data
model_df = pd.merge(
    model_df, 
    doses_long[["Date", "state", "at_least_3_doses"]], 
    how="left", 
    left_on=["endtime", "state"],    
    right_on=["Date", "state"]       
)
model_df.drop(columns=["Date"], inplace=True)
model_df["at_least_3_doses"] = model_df["at_least_3_doses"].fillna(0)
# Merge daily cases data
model_df = pd.merge(
    model_df, 
    cases_long[["Date", "state", "daily_cases"]], 
    how="left", 
    left_on=["endtime", "state"], 
    right_on=["Date", "state"]       
)
model_df.drop(columns=["Date"], inplace=True)



Use state and response time to merge 'during_mandate' variagbe & convert unordered cate variables into dummy variables

In [14]:
mandate_lookup_df = pd.read_csv(OUTPUT_DIR / "mandate_start_dates.csv")
mandate_lookup_df["Date"] = pd.to_datetime(mandate_lookup_df["Date"])

states_date = {}
for state, date in zip(mandate_lookup_df["RegionName"], mandate_lookup_df["Date"]):
    states_date.update({state: [date]})

def mandates_convert(row):
    endtime = pd.to_datetime(row["endtime"])
    state = row["state"]
    if state not in states_date:
        return 0
    if states_date[state][0] <= endtime:
        return 1
    else:
        return 0
model_df["during_mandate"] = model_df.apply(mandates_convert, axis=1)


exp_cols = ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4", "comorbidity_status"]

# String Version
model_df_str = model_df.copy()
model_df_str[exp_cols] = model_df_str[exp_cols].fillna("N/A")

#NaN Version
model_df_nan = model_df.copy()

#drop version
model_df_drop = model_df.dropna(subset=exp_cols).copy()

categorical_features = [
    "PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4",
    "WCRex1", "WCRex2", 
    "comorbidity_status", 
    "employment_status", 
    "gender", 
    "i11_health", "i9_health", 
    "state"
]

def process_dummies(df, is_nan_version=False):
    df_out = df.copy()
    for col in categorical_features:
        dummy = pd.get_dummies(df_out[col], prefix=col, drop_first=True, dtype=float)
        
        # Core logic: If in the NaN version, to allow XGBoost to recognize missing values, we must force rows that are all 0s back to np.nan.
        if is_nan_version and col in exp_cols:
            dummy.loc[df_out[col].isna(), dummy.columns] = np.nan
            
        df_out = pd.concat([df_out, dummy], axis=1)
        df_out = df_out.drop(col, axis=1)
    return df_out

model_df.to_csv(OUTPUT_DIR / "processed_data_merged.csv", index=False)
model_df.head()

processed_str = process_dummies(model_df_str, is_nan_version=False)
processed_nan = process_dummies(model_df_nan, is_nan_version=True)
processed_drop = process_dummies(model_df_drop, is_nan_version=False)

# 5.3.2 - 5.3.3 Data Splitting and Construction of Predictive Scenarios
Firstly, the data was split into training and testing sets in an 80:20 ratio stratified with the value of during_mandate. Than, data was divided into two parts: predicting mask wearing and predicting general. The data splitting for imputation experiment is also in this section.
protective behaviour.

In [15]:
def split_and_save(df, version_name, target_col):
    df_copy = df.copy()
    df_copy["endtime"] = pd.to_datetime(df_copy["endtime"])
    
    mandate_period_label = df_copy.loc[:, "during_mandate"]
    
    df_train, df_test = train_test_split(
        df_copy,
        test_size=0.2,
        random_state=1935487,
        stratify=mandate_period_label,
    )
    
    if target_col == 'face_mask_bi':
        leakage_cols = [
            'endtime', 'face_mask_scale', 
            'protective_behaviour_bi', 'protective_behaviour_scale',
            'RecordNo' 
        ]
    elif target_col == 'protective_behaviour_bi':
        leakage_cols = [
            'endtime', 'protective_behaviour_scale',
            'protective_behaviour_nomask_bi', 'protective_behaviour_nomask_scale',
            'face_mask_bi', 'face_mask_scale', 
            'RecordNo' 
        ]
    
    X_train = df_train.drop(columns=[target_col] + leakage_cols, errors='ignore')
    y_train = df_train[target_col]
    X_test = df_test.drop(columns=[target_col] + leakage_cols, errors='ignore')
    y_test = df_test[target_col]
    
    target_name = target_col.replace('_bi', '')
    X_train.to_csv(OUTPUT_DIR / f"X_train_{version_name}_{target_name}.csv", index=False)
    y_train.to_csv(OUTPUT_DIR / f"y_train_{version_name}_{target_name}.csv", index=False)
    X_test.to_csv(OUTPUT_DIR / f"X_test_{version_name}_{target_name}.csv", index=False)
    y_test.to_csv(OUTPUT_DIR / f"y_test_{version_name}_{target_name}.csv", index=False)
    
    print(f"[{version_name} - {target_name}] Train shape: {X_train.shape}")


# Data splitting for imputation experiment
targets = ['face_mask_bi', 'protective_behaviour_bi']
for target in targets:
    split_and_save(processed_nan, "exp_nan", target)
    split_and_save(processed_str, "exp_str", target)
    split_and_save(processed_drop, "exp_drop", target)
print(processed_str.isna().sum())

[exp_nan - face_mask] Train shape: (35848, 62)
[exp_str - face_mask] Train shape: (35848, 62)
[exp_drop - face_mask] Train shape: (35521, 62)
[exp_nan - protective_behaviour] Train shape: (35848, 60)
[exp_str - protective_behaviour] Train shape: (35848, 60)
[exp_drop - protective_behaviour] Train shape: (35521, 60)
RecordNo                      0
endtime                       0
i2_health                  2002
age                           0
household_size             1123
                           ... 
state_Queensland              0
state_South Australia         0
state_Tasmania                0
state_Victoria                0
state_Western Australia       0
Length: 68, dtype: int64


## 5.4 Models


### Model Definition and Evaluation Pipeline
The next chunk constructs an imblearn pipeline, performs GridSearchCV with automated oversampling, and saves the final refitted model on AUC.
Due to the fact that many details mentioned in the article need to be defined in the same pipeline, this section of code includes the following parts:
>- 5.3.1 Class Imbalance and Resampling
>- 5.3.2 Cross Validation
>- 5.5.1 Evaluation Metrics


In [16]:
SEED = 1935487
METRIC_LIST = ['roc_auc', 'precision', 'recall', 'accuracy', 'f1']

# 5-Fold CV
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def run_model(model_number, target_name, model_type, clf, param_grid):

    print(f"[{target_name}] {model_number}: {model_type}")
    
    X_train = pd.read_csv(OUTPUT_DIR / f"X_train_{model_number}_{target_name}.csv")
    y_train = pd.read_csv(OUTPUT_DIR / f"y_train_{model_number}_{target_name}.csv").values.ravel()
    X_train = X_train.replace({'Yes': 1, 'No': 0})
    
    for col in X_train.select_dtypes(include=['object']).columns:
        X_train[col] = X_train[col].astype('category')

    # Use LabelEncoder to convert Yes/No to 1/0 to prevent errors in  XGBoost
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    imputer = SimpleImputer(strategy='median')
    if model_type == "MLP":
        X_train = X_train.astype(np.float32)
        y_train = y_train.astype(np.float32).reshape(-1, 1)
        clf.set_params(module__input_dim=X_train.shape[1])
    
    steps = [
        ('imputer', SimpleImputer(strategy='median')),
        # 5.3.1 Resampling
        ('sampler', RandomOverSampler(random_state=SEED))
    ]
    
    if model_type == "MLP":
        steps.append(('scaler', StandardScaler()))
        
    steps.append(('classifier', clf))
    
    pipeline = ImbPipeline(steps)
    grid_params = {f"classifier__{k}": v for k, v in param_grid.items()}
    grid_search = GridSearchCV(
        pipeline, 
        param_grid=grid_params, 
        cv=cv_strategy,
        scoring=METRIC_LIST, 
        refit='roc_auc', 
        n_jobs=-3
    )
    grid_search.fit(X_train, y_train)
    
    # Extract performance metrics for the best model index
    best_idx = grid_search.best_index_
    best_params = grid_search.best_params_
    
    
    def get_mean_se_string(metric):
        mean_val = grid_search.cv_results_[f'mean_test_{metric}'][best_idx]
        std_val = grid_search.cv_results_[f'std_test_{metric}'][best_idx]
        se_val = std_val / np.sqrt(cv_strategy.get_n_splits())
        return f"{mean_val:.3f} ({se_val:.3f})"
    
    metrics = {
        "model_number": model_number,
        "model_type": model_type,
        "roc_auc": get_mean_se_string('roc_auc'),
        "precision": get_mean_se_string('precision'),
        "recall": get_mean_se_string('recall'),
        "accuracy": get_mean_se_string('accuracy'),
        "f1": get_mean_se_string('f1')
    }
    clean_params = {k.replace('classifier__', ''): v for k, v in best_params.items()}
    print(f" Best Params:{clean_params}")
    print(f" Mean ROC AUC of cv:{metrics['roc_auc']}")
    
    with open(RESULTS_DIR / f"{model_number}_{target_name}_{model_type}.pkl", "wb") as f:
        pickle.dump(grid_search.cv_results_, f)
        
    with open(MODEL_DIR / f"{model_number}_{target_name}_{model_type}_final.pkl", "wb") as f:
        pickle.dump(grid_search.best_estimator_, f)
        
    return metrics, grid_search.best_estimator_

### 5.4.1 - 5.4.4 Defination for machine learning models

In [17]:
#Logistic Regression
grid_lr = {
    'C': [0.01, 0.1, 1, 10], 
    'penalty': ['l2']
}
lr_clf = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED)



# Classification Trees
grid_dt = {
    'max_depth': [5, 10, 15, 20, None], 
    'min_samples_leaf': [1, 2, 5, 10], 
    'min_samples_split': [2, 5, 10] 
}
dt_clf = DecisionTreeClassifier(random_state=SEED)


#XGBoost
grid_xgb = {
    'learning_rate': [0.05, 0.1, 0.2], 
    'n_estimators': [100, 250, 500], 
    'max_depth': [4, 6, 8], 
    'colsample_bytree': [0.6, 0.8, 1.0]     
}
xgb_clf = XGBClassifier( eval_metric='logloss', random_state=SEED, enable_categorical=True)

### 5.4.5 Defination for MLPs

In [18]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_size_1=50, hidden_size_2=25, hidden_size_3=10, dropout_rate=0.2, activation=nn.ReLU): 
        super().__init__()
        
        self.hidden_size_2 = hidden_size_2
        self.hidden_size_3 = hidden_size_3
        
        # First hidden layer (always present)
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_size_1),
            activation(),
        )
        
        # Track the output dimension of the last active layer
        last_out_dim = hidden_size_1
        
        # Second hidden layer (conditionally present)
        if self.hidden_size_2 > 0:
            self.layer2 = nn.Sequential(
                nn.Linear(last_out_dim, hidden_size_2),
                activation(),
            )
            last_out_dim = hidden_size_2
            
            # Third hidden layer (only evaluated if layer 2 exists and size_3 > 0)
            if self.hidden_size_3 > 0:
                self.layer3 = nn.Sequential(
                    nn.Linear(last_out_dim, hidden_size_3),
                    activation(),
                )
                last_out_dim = hidden_size_3
        self.dropout = nn.Dropout(dropout_rate)
                
        # Output layer 
        self.output_layer = nn.Linear(last_out_dim, 1)

    def forward(self, X):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(np.array(X), dtype=torch.float32)
            
        out = self.layer1(X)
        
        # Data flows through layer 2 and layer 3 only if they exist
        if self.hidden_size_2 > 0:
            out = self.layer2(out)
            if self.hidden_size_3 > 0:
                out = self.layer3(out)
        out = self.dropout(out)
                
        return self.output_layer(out)


n_features = 61 

mlp_clf = NeuralNetBinaryClassifier(
    module=MLP,
    module__input_dim=n_features,
    module__hidden_size_1=50,  
    module__hidden_size_2=25,
    module__hidden_size_3=10, 
    module__dropout_rate=0.2,
    module__activation=nn.ReLU,
    criterion=nn.BCEWithLogitsLoss, 
    optimizer=torch.optim.Adam,
    max_epochs=50,
    verbose=0,
    callbacks=[
        EarlyStopping(
            monitor='valid_loss',
            patience=5,
            lower_is_better=True
        )
    ]
)

# grid search space
grid_mlp = {
    'lr': [0.0005, 0.001, 0.005],
    
    # Layer 1 (Always exists)
    'module__hidden_size_1': [50, 64,128],
    
    # Layer 2 (0 means 1-layer network)
    'module__hidden_size_2': [0, 16, 25, 50], 
    
    # Layer 3 (0 means max 2-layer network)
    'module__hidden_size_3': [0, 8, 16], 
    'module__activation': [nn.ReLU, nn.LeakyReLU, nn.GELU, nn.Tanh],
    
    'module__dropout_rate': [0.3, 0.4, 0.5],
    'optimizer__weight_decay': [0.0, 1e-4]
}

# metrics_mlp = run_model("exp_str", "MLP", mlp_clf, grid_mlp)

### 6.2.1 Imputation experiment
Here, we investigate how different missing data strategies affect the model's predictive performance. Specifically, we compare the results of leaving structural missing values as `np.nan` against filling them with an explicit string category. And also just remove these columns. The data splitting for all scenarios was completed in the past steps.

In [19]:
metrics_nan, _ = run_model("exp_nan", "face_mask", "XGBoost", xgb_clf, grid_xgb)
metrics_str, _ = run_model("exp_str", "face_mask", "XGBoost", xgb_clf, grid_xgb)
metrics_drop, _ = run_model("exp_drop", "face_mask", "XGBoost", xgb_clf, grid_xgb)

print(f"NaN version ROC AUC: {metrics_nan['roc_auc']}")
print(f"Str version ROC AUC: {metrics_str['roc_auc']}")
print(f"Drop version ROC AUC: {metrics_drop['roc_auc']}")

[face_mask] exp_nan: XGBoost


C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\4020533268.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = X_train.replace({'Yes': 1, 'No': 0})


 Best Params:{'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 250}
 Mean ROC AUC of cv:0.911 (0.001)
[face_mask] exp_str: XGBoost


C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\4020533268.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = X_train.replace({'Yes': 1, 'No': 0})


 Best Params:{'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 250}
 Mean ROC AUC of cv:0.911 (0.001)
[face_mask] exp_drop: XGBoost


C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\4020533268.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = X_train.replace({'Yes': 1, 'No': 0})


 Best Params:{'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 500}
 Mean ROC AUC of cv:0.912 (0.002)
NaN version ROC AUC: 0.911 (0.001)
Str version ROC AUC: 0.911 (0.001)
Drop version ROC AUC: 0.912 (0.002)


Evaluate saved models on their corresponding test sets

In [20]:
for exp_name in ["exp_nan", "exp_str", "exp_drop"]:
    
    X_test = pd.read_csv(OUTPUT_DIR / f"X_test_{exp_name}_face_mask.csv")
    y_test = LabelEncoder().fit_transform(pd.read_csv(OUTPUT_DIR / f"y_test_{exp_name}_face_mask.csv").values.ravel())

    X_test = X_test.replace({'Yes': 1, 'No': 0})
    
    for c in X_test.select_dtypes(include=['object']).columns: X_test[c] = X_test[c].astype('category')
        
    with open(MODEL_DIR / f"{exp_name}_face_mask_XGBoost_final.pkl", "rb") as f:
        best_model = pickle.load(f)
    y_pred, y_prob = best_model.predict(X_test), best_model.predict_proba(X_test)[:, 1]
    
    print(f"{exp_name} (face_mask) AUC: {roc_auc_score(y_test, y_prob):.3f}")


exp_nan (face_mask) AUC: 0.914
exp_str (face_mask) AUC: 0.914
exp_drop (face_mask) AUC: 0.912


C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\830225563.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test = X_test.replace({'Yes': 1, 'No': 0})
C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\830225563.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test = X_test.replace({'Yes': 1, 'No': 0})
C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\830225563.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.

### 6.2.2 Run four models and choose the best

In [21]:
models_dict = {
    "LogisticRegression": (lr_clf, grid_lr),
    "DecisionTree": (dt_clf, grid_dt),
    "XGBoost": (xgb_clf, grid_xgb),
    "MLP": (mlp_clf, grid_mlp)
}
final_summary = {}
best_models_dict = {} 
targets = ['face_mask_bi', 'protective_behaviour_bi']

for target in targets:
    target_name = target.replace('_bi', '')
    print(f"\n{target_name}")
    task_summary = {}
    best_auc = 0.0
    best_model_name = ""
    best_pipeline = None
    cv_metrics_list = []
    # Grid search all models for current task
    for m_name, (clf, grid) in models_dict.items():
        
        # Load task-specific data within run_model
        metrics, trained_pipe = run_model("exp_str", target_name, m_name, clf, grid)
        auc_val = float(metrics["roc_auc"].split()[0])
        task_summary[m_name] = auc_val

        cv_metrics_list.append({
            "Model": m_name,
            "AUC-ROC": metrics["roc_auc"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1 Score": metrics["f1"]
        })
        
        # Lock the best model architecture
        if auc_val > best_auc:
            best_auc = auc_val
            best_model_name = m_name
            best_pipeline = trained_pipe
    final_summary[target_name] = task_summary
    best_models_dict[target_name] = (best_model_name, best_pipeline)
    print(f"Best {target_name} Model: {best_model_name} (AUC: {best_auc:.3f})")
    display(pd.DataFrame(cv_metrics_list))


face_mask
[face_mask] exp_str: LogisticRegression


C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\4020533268.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = X_train.replace({'Yes': 1, 'No': 0})
c:\Users\ljj15\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\ljj15\Ap

 Best Params:{'C': 0.01, 'penalty': 'l2'}
 Mean ROC AUC of cv:0.796 (0.001)
[face_mask] exp_str: DecisionTree
 Best Params:{'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 2}
 Mean ROC AUC of cv:0.877 (0.002)
[face_mask] exp_str: XGBoost


C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\4020533268.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = X_train.replace({'Yes': 1, 'No': 0})


 Best Params:{'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 250}
 Mean ROC AUC of cv:0.911 (0.001)
[face_mask] exp_str: MLP


...

,Model,AUC-ROC,Precision,Recall,F1 Score
0,LogisticRegression,0.796 (0.001),0.741 (0.001),0.767 (0.002),0.754 (0.001)
1,DecisionTree,0.877 (0.002),0.834 (0.003),0.808 (0.003),0.821 (0.001)
2,XGBoost,0.911 (0.001),0.857 (0.002),0.846 (0.002),0.851 (0.002)
3,MLP,0.882 (0.001),0.865 (0.002),0.730 (0.003),0.792 (0.002)



protective_behaviour
[protective_behaviour] exp_str: LogisticRegression


c:\Users\ljj15\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


 Best Params:{'C': 0.1, 'penalty': 'l2'}
 Mean ROC AUC of cv:0.721 (0.001)
[protective_behaviour] exp_str: DecisionTree
 Best Params:{'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 2}
 Mean ROC AUC of cv:0.768 (0.001)
[protective_behaviour] exp_str: XGBoost
 Best Params:{'colsample_bytree': 0.6, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 500}
 Mean ROC AUC of cv:0.832 (0.001)
[protective_behaviour] exp_str: MLP
 Best Params:{'lr': 0.0005, 'module__activation': <class 'torch.nn.modules.activation.GELU'>, 'module__dropout_rate': 0.5, 'module__hidden_size_1': 128, 'module__hidden_size_2': 0, 'module__hidden_size_3': 16, 'optimizer__weight_decay': 0.0}
 Mean ROC AUC of cv:0.789 (0.002)
Best protective_behaviour Model: XGBoost (AUC: 0.832)


,Model,AUC-ROC,Precision,Recall,F1 Score
0,LogisticRegression,0.721 (0.001),0.774 (0.002),0.697 (0.007),0.734 (0.003)
1,DecisionTree,0.768 (0.001),0.809 (0.002),0.697 (0.005),0.749 (0.002)
2,XGBoost,0.832 (0.001),0.829 (0.001),0.791 (0.003),0.809 (0.002)
3,MLP,0.789 (0.002),0.872 (0.002),0.551 (0.003),0.676 (0.003)


### 6.2.2 Test Set Evaluation
In this section, the best performing model of the whole  will be evaluate through testing set

In [22]:
targets = ['face_mask_bi', 'protective_behaviour_bi']
test_results = []

for target in targets:
    target_name = target.replace('_bi', '')
    best_model_name, best_pipeline = best_models_dict[target_name] 
    
    X_test_raw = pd.read_csv(OUTPUT_DIR / f"X_test_exp_str_{target_name}.csv")
    y_test = LabelEncoder().fit_transform(pd.read_csv(OUTPUT_DIR / f"y_test_exp_str_{target_name}.csv").values.ravel())
    X_test_raw = X_test_raw.replace({'Yes': 1, 'No': 0})
    if best_model_name == "MLP":
        X_test_raw = X_test_raw.astype(np.float32)

    # Predict
    y_pred = best_pipeline.predict(X_test_raw)
    y_prob = best_pipeline.predict_proba(X_test_raw)[:, 1]

    # Save metrics
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    test_metrics = {
        "roc_auc": roc_auc_score(y_test, y_prob),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "classification_report": classification_report(y_test, y_pred, output_dict=True)
    }
    
    test_results.append({
        "Target": target_name,
        "Best Model": best_model_name,
        "AUC-ROC": test_metrics["roc_auc"],
        "Precision": test_metrics["precision"],
        "Recall": test_metrics["recall"],
        "F1 Score": test_metrics["f1"]
    })
    
    with open(RESULTS_DIR / f"{best_model_name}_{target_name}_test_evaluation.pkl", "wb") as f:
        pickle.dump(test_metrics, f)
        
    print(f"[{target_name}] Test AUC: {test_metrics['roc_auc']:.3f} (Saved to {RESULTS_DIR})")

[face_mask] Test AUC: 0.914 (Saved to results)
[protective_behaviour] Test AUC: 0.838 (Saved to results)


C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\1994304552.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_raw = X_test_raw.replace({'Yes': 1, 'No': 0})


### 6.2.3 SHAP Value Analysis & Data Export

In [23]:
for target in targets:
    target_name = target.replace('_bi', '')
    best_model_name, best_pipeline = best_models_dict[target_name]
    print(f"Extracting SHAP for: {target_name} ({best_model_name})...")
    
    # Reload and format feature matrix for SHAP
    X_test_raw = pd.read_csv(OUTPUT_DIR / f"X_test_exp_str_{target_name}.csv")
    feature_names = X_test_raw.columns.tolist()
    
    X_test_raw = X_test_raw.replace({'Yes': 1, 'No': 0})
    
    imputer = best_pipeline.named_steps['imputer']
    X_test_processed = imputer.transform(X_test_raw)
    if 'scaler' in best_pipeline.named_steps:
        scaler = best_pipeline.named_steps['scaler']
        X_test_processed = scaler.transform(X_test_processed)

    if best_model_name == "MLP":
        X_test_processed = X_test_processed.astype(np.float32)
        
    classifier = best_pipeline.named_steps['classifier']
    X_test_shap = pd.DataFrame(X_test_processed, columns=feature_names)

    # Compute SHAP
    if best_model_name in ["XGBoost", "DecisionTree"]:
        explainer = shap.TreeExplainer(classifier)
        shap_values = explainer.shap_values(X_test_shap)
        expected_value = explainer.expected_value
    else:  
        background = shap.kmeans(X_test_shap, 50)
        def mlp_predict_proba(x):
            return classifier.predict_proba(torch.tensor(x, dtype=torch.float32))[:, 1]
        explainer = shap.KernelExplainer(mlp_predict_proba, background)
        X_test_shap = X_test_shap.iloc[:500]  
        shap_values = explainer.shap_values(X_test_shap)
        expected_value = explainer.expected_value

    # Standardize output formats
    if isinstance(shap_values, list) and len(shap_values) == 2:
        shap_values = shap_values[1]
    if isinstance(expected_value, list) and len(expected_value) == 2:
        expected_value = expected_value[1]

    shap_data = {
        "shap_values": shap_values,
        "expected_value": expected_value,
        "X_test_features": X_test_shap,
        "feature_names": feature_names
    }

    # Save to disk
    with open(RESULTS_DIR / f"{best_model_name}_{target_name}_shap_data.pkl", "wb") as f:
        pickle.dump(shap_data, f)
        
    print(f"SHAP data for {target_name} saved")

C:\Users\ljj15\AppData\Local\Temp\ipykernel_15944\3309265039.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_raw = X_test_raw.replace({'Yes': 1, 'No': 0})


Extracting SHAP for: face_mask (XGBoost)...
SHAP data for face_mask saved
Extracting SHAP for: protective_behaviour (XGBoost)...
SHAP data for protective_behaviour saved
